# 14 - Error Analysis

## Context
Model machine learning hampir tidak pernah sempurna. Menganalisis kesalahan prediksi model — baik itu False Positives (transaksi normal yang dianggap fraud) maupun False Negatives (transaksi fraud yang lolos) — sangat penting untuk mengetahui kelemahan model dan mencari ide perbaikan fitur selanjutnya.

## Objective
- Melatih model untuk melakukan prediksi pada data validasi.
- Mengidentifikasi transaksi yang salah diprediksi oleh model.
- Profiling kesalahan berdasarkan nominal transaksi (`TransactionAmt`) dan tingkat keyakinan model.

### Impor Library

In [ ]:
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
import lightgbm as lgb

warnings.filterwarnings("ignore")
plt.style.use("seaborn-v0_8-darkgrid")

### Muat Dataset Terproses & Pemisahan Train/Val

In [ ]:
path = Path("../data/processed/train_features.parquet")
if not path.exists():
    path = Path("../data/interim/train_merged.parquet")
train = pd.read_parquet(path)

exclude_cols = ["TransactionID", "isFraud", "TransactionDT"]
feature_cols = [c for c in train.select_dtypes(include="number").columns if c not in exclude_cols]
X = train[feature_cols]
y = train["isFraud"]

X_tr, X_va, y_tr, y_va = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

### Latih Model & Lakukan Prediksi

In [ ]:
# Latih model LightGBM base untuk analisis error
dtrain = lgb.Dataset(X_tr, label=y_tr)
dval = lgb.Dataset(X_va, label=y_va, reference=dtrain)
model = lgb.train(
    {"objective": "binary", "metric": "auc", "num_leaves": 31, "learning_rate": 0.1, "verbose": -1, "random_state": 42},
    dtrain, num_boost_round=100, valid_sets=[dval], callbacks=[lgb.early_stopping(15, verbose=False)]
)

preds = model.predict(X_va, num_iteration=model.best_iteration)
pred_class = (preds >= 0.5).astype(int)

### Identifikasi Kelas Kesalahan (False Positives & False Negatives)

In [ ]:
analysis_df = X_va.copy()
analysis_df["true_label"] = y_va
analysis_df["pred_prob"] = preds
analysis_df["pred_label"] = pred_class

# Pisahkan data transaksi yang salah prediksi
false_positives = analysis_df[(analysis_df["true_label"] == 0) & (analysis_df["pred_label"] == 1)]
false_negatives = analysis_df[(analysis_df["true_label"] == 1) & (analysis_df["pred_label"] == 0)]

print(f"Total Sampel Validasi: {len(X_va):,}")
print(f"False Positives (Normal dianggap Fraud): {len(false_positives):,}")
print(f"False Negatives (Fraud lolos dari Model):  {len(false_negatives):,}")

### Analisis Nominal Transaksi pada Kelompok Error

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
sns.kdeplot(analysis_df[analysis_df["true_label"] == 0]["TransactionAmt"], label="Normal (Sebenarnya)", log_scale=True, ax=ax)
sns.kdeplot(false_positives["TransactionAmt"], label="False Positives (Salah Tuduh)", log_scale=True, ax=ax)
sns.kdeplot(false_negatives["TransactionAmt"], label="False Negatives (Fraud Lolos)", log_scale=True, ax=ax)
ax.set_title("Distribusi Nominal Transaksi pada Tiap Kelompok Error")
ax.set_xlabel("Nominal Transaksi (USD, skala log)")
ax.legend()
plt.show()

### Top 10 Kasus Terparah: False Positives dengan Keyakinan Tinggi

In [ ]:
print("Top 10 Transaksi Normal yang Dicurigai Kuat oleh Model sebagai Fraud:")
print(false_positives.sort_values("pred_prob", ascending=False)[["TransactionAmt", "card1", "pred_prob"]].head(10))

### Top 10 Kasus Terparah: False Negatives dengan Keyakinan Tinggi

In [ ]:
print("Top 10 Transaksi Fraud yang Dilewatkan Model dengan Keyakinan Tinggi (Sangat Normal bagi Model):")
print(false_negatives.sort_values("pred_prob", ascending=True)[["TransactionAmt", "card1", "pred_prob"]].head(10))